# 03 – Experiments (R6)

In this notebook we test the two hypotheses from R5, using the feature table
from 02_features.ipynb.

- **H1:** If referred students fail more because they were already struggling,
  the gap should go away when we compare students with similar prior grades
  and term 1 results.
- **H2:** If attending tutoring helps, referred students who attended more
  should fail less.

The experiments are comparisons, not models, so they use all 120,000 rows.

**Rule from R5:** groups with fewer than 50 student-subject pairs are left out
when deciding. We apply this to each side of a comparison (for example, at
least 50 referred and 50 non-referred pairs in a group).

## Step 1 – Load the data

In [1]:
# Let the notebook find the src folder
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.split import load_features

# Rule from R5: groups smaller than this are left out of the decision
MIN_N = 50

# Order of the GPA groups in tables
GPA_ORDER = ["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)", "Missing"]

# Load the feature table
df = load_features()
print("All rows:", len(df))

All rows: 120000


**Result:** The feature table loaded with 120,000 rows.

## Step 2 – Confidence intervals
A failure rate from a small group is less certain than one from a big group.
For each rate we show a 95% confidence interval: the range where the true rate
probably is. We use the Wilson method, which works well for rates.

In [2]:
# 95% confidence interval for a rate (k failures out of n), Wilson method
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (max(0.0, center - half), min(1.0, center + half))

# Quick check: 10 failures out of 100 should give about 5.5% to 17.4%
low, high = wilson_ci(10, 100)
print(f"10 of 100: {low:.1%} to {high:.1%}")

10 of 100: 5.5% to 17.4%


**Result:** The helper works: 10 failures out of 100 gives a 95% interval of
about 5.5% to 17.4%.

## Step 3 – H1: Were referred students already struggling?

Prior struggle can act as a hidden variable (a **fork**): it affects both
being referred and failing, so the overall comparison can be misleading.

**Experiment:**
- **Held fixed:** prior GPA group (4 quartiles + "Missing") and term 1 result
  (passed / failed), giving 10 groups
- **Varied:** referred before term 3 vs. not referred
- **Measured:** term 3 failure rate, with a 95% confidence interval
- **Left out:** pairs first referred during term 3 (1,379), as R5 says

**Decision (R5):** H1 is wrong if referred students fail more often than
non-referred students in most of the groups used.

In [3]:
# Leave out pairs first referred during term 3 (R5)
h1 = df[~df["referred_during_t3"]].copy()
print("Rows for H1:", len(h1), "| left out:", len(df) - len(h1))

Rows for H1: 118621 | left out: 1379


In [4]:
# The overall comparison (the paradox)
overall = h1.groupby("referred_before_t3")["failed_t3"].agg(n="size", rate="mean")
print("\nOverall term 3 failure rate:")
print(overall)


Overall term 3 failure rate:
                         n      rate
referred_before_t3                  
False               105083  0.088587
True                 13538  0.195524


In [5]:
# Count pairs and failures in each group, for referred and not referred
h1["side"] = np.where(h1["referred_before_t3"], "referred", "not_referred")
counts = (h1.groupby(["gpa_group", "failed_t1", "side"], observed=True)["failed_t3"]
            .agg(n="size", fails="sum")
            .unstack("side"))
counts.columns = [f"{stat}_{side}" for stat, side in counts.columns]
counts = counts.fillna(0).astype(int).reset_index()

In [6]:
# Failure rates and the difference (referred minus not referred)
counts["rate_referred"] = counts["fails_referred"] / counts["n_referred"]
counts["rate_not_referred"] = counts["fails_not_referred"] / counts["n_not_referred"]
counts["difference"] = counts["rate_referred"] - counts["rate_not_referred"]

In [7]:
# 95% confidence intervals for each rate
counts["ci_referred"] = [f"{lo:.1%} to {hi:.1%}" for lo, hi in
                         (wilson_ci(k, n) for k, n in zip(counts["fails_referred"], counts["n_referred"]))]
counts["ci_not_referred"] = [f"{lo:.1%} to {hi:.1%}" for lo, hi in
                             (wilson_ci(k, n) for k, n in zip(counts["fails_not_referred"], counts["n_not_referred"]))]

In [8]:
# R5 rule: use a group only if both sides have at least 50 pairs
counts["used"] = (counts["n_referred"] >= MIN_N) & (counts["n_not_referred"] >= MIN_N)


In [9]:
# Sort the groups in a readable order
counts["gpa_group"] = pd.Categorical(counts["gpa_group"], categories=GPA_ORDER, ordered=True)
counts = counts.sort_values(["gpa_group", "failed_t1"]).reset_index(drop=True)

In [10]:
# Show the table
h1_table = counts[["gpa_group", "failed_t1",
                   "n_referred", "rate_referred", "ci_referred",
                   "n_not_referred", "rate_not_referred", "ci_not_referred",
                   "difference", "used"]]
print(h1_table.to_string(formatters={"rate_referred": "{:.1%}".format,
                                      "rate_not_referred": "{:.1%}".format,
                                      "difference": "{:+.1%}".format}))

      gpa_group  failed_t1  n_referred rate_referred     ci_referred  n_not_referred rate_not_referred ci_not_referred difference   used
0   Q1 (lowest)      False        5270         15.7%  14.8% to 16.7%           13310             20.4%  19.7% to 21.1%      -4.7%   True
1   Q1 (lowest)       True        3256         51.2%  49.4% to 52.9%            6868             55.2%  54.0% to 56.3%      -4.0%   True
2            Q2      False        2687          3.0%    2.4% to 3.7%           23411              6.0%    5.7% to 6.3%      -3.0%   True
3            Q2       True         199         18.6%  13.8% to 24.6%            2154             25.5%  23.7% to 27.4%      -6.9%   True
4            Q3      False        1317          0.8%    0.4% to 1.4%           27109              1.3%    1.2% to 1.5%      -0.6%   True
5            Q3       True          16          6.2%   1.1% to 28.3%             488             14.3%  11.5% to 17.7%      -8.1%  False
6  Q4 (highest)      False         414   

In [11]:
# Decision (R5): wrong if referred fail more in most of the used groups
used = counts[counts["used"]]
higher = int((used["rate_referred"] > used["rate_not_referred"]).sum())
print(f"\nGroups used: {len(used)} of {len(counts)}")
print(f"Groups where referred students fail MORE: {higher} of {len(used)}")
print("H1 verdict:", "REJECTED" if higher > len(used) / 2 else "SUPPORTED")


Groups used: 7 of 10
Groups where referred students fail MORE: 0 of 7
H1 verdict: SUPPORTED


**Result:**

**Overall (the paradox):** referred students failed term 3 more often than
non-referred students: 19.6% (of 13,538) vs. 8.9% (of 105,083).

**Inside groups:** in all 7 groups with at least 50 pairs on each side,
referred students failed **less** often than non-referred students with the
same prior GPA group and term 1 result. For example, among Q1 students who
passed term 1, 15.7% of referred students failed vs. 20.4% of non-referred
students, and the 95% intervals do not overlap (14.8–16.7% vs. 19.7–21.1%).
The 3 small groups that were left out point the same way.

**Verdict: H1 is supported.** Referred students fail more overall only because
they started out weaker. This is the fork from R5: prior struggle affects
both being referred and failing. Once we compare students who started at the
same level, the gap goes away and even reverses.

**Caution:** this shows that the overall gap comes from who was referred. It
does not prove that tutoring caused the lower failure inside groups, because
referred students may differ in other ways we can't see (for example,
motivation or parent support).

## Step 4 – H2: Do students who attend more fail less?

**H2 (from R5):** if attending tutoring helps, referred students who attended
more should fail less.

**Experiment:**
- **Students:** referred before term 3, with at least one session scheduled
  before term 3 (13,538 pairs)
- **Held fixed:** prior GPA group (4 quartiles + "Missing")
- **Varied:** attendance group before term 3 (0–25%, above 25–50%,
  above 50–75%, above 75–100%)
- **Measured:** term 3 failure rate, with a 95% confidence interval

**Decision (R5):** H2 is wrong if the failure rate does not go down as
attendance rises in most of the GPA groups used. We count a GPA group as
"going down" if failure is lower in each higher attendance group than in the
one before it, using only attendance groups with at least 50 pairs.

In [19]:
# Referred students with at least one session before term 3
h2 = df[df["referred_before_t3"] & (df["sessions_scheduled_before_t3"] > 0)].copy()
print("Rows for H2:", len(h2))

ATT_ORDER = ["0-25%", ">25-50%", ">50-75%", ">75-100%"]

Rows for H2: 13538


In [20]:

# Overall: failure rate by attendance group (all GPA groups together)
overall_h2 = h2.groupby("attendance_group")["failed_t3"].agg(n="size", rate="mean").reindex(ATT_ORDER)
print("\nOverall term 3 failure rate by attendance:")
print(overall_h2.to_string(formatters={"rate": "{:.1%}".format}))


Overall term 3 failure rate by attendance:
                     n  rate
attendance_group            
0-25%             1889 31.7%
>25-50%           3657 24.7%
>50-75%           4273 16.7%
>75-100%          3719 11.6%


In [21]:
# Inside each GPA group: count pairs and failures per attendance group
t = (h2.groupby(["gpa_group", "attendance_group"], observed=True)["failed_t3"]
       .agg(n="size", fails="sum").reset_index())
t["rate"] = t["fails"] / t["n"]
t["ci"] = [f"{lo:.1%} to {hi:.1%}" for lo, hi in (wilson_ci(k, n) for k, n in zip(t["fails"], t["n"]))]
t["used"] = t["n"] >= MIN_N

In [22]:
# Sort in a readable order
t["gpa_group"] = pd.Categorical(t["gpa_group"], categories=GPA_ORDER, ordered=True)
t["attendance_group"] = pd.Categorical(t["attendance_group"], categories=ATT_ORDER, ordered=True)
t = t.sort_values(["gpa_group", "attendance_group"]).reset_index(drop=True)
print("\n" + t.to_string(formatters={"rate": "{:.1%}".format}))


       gpa_group attendance_group     n  fails  rate              ci  used
0    Q1 (lowest)            0-25%  1251    554 44.3%  41.6% to 47.1%  True
1    Q1 (lowest)          >25-50%  2365    842 35.6%  33.7% to 37.6%  True
2    Q1 (lowest)          >50-75%  2689    683 25.4%  23.8% to 27.1%  True
3    Q1 (lowest)         >75-100%  2221    416 18.7%  17.2% to 20.4%  True
4             Q2            0-25%   378     31  8.2%   5.8% to 11.4%  True
5             Q2          >25-50%   773     50  6.5%    4.9% to 8.4%  True
6             Q2          >50-75%   925     24  2.6%    1.7% to 3.8%  True
7             Q2         >75-100%   810     13  1.6%    0.9% to 2.7%  True
8             Q3            0-25%   148      2  1.4%    0.4% to 4.8%  True
9             Q3          >25-50%   315      7  2.2%    1.1% to 4.5%  True
10            Q3          >50-75%   413      2  0.5%    0.1% to 1.7%  True
11            Q3         >75-100%   457      0  0.0%    0.0% to 0.8%  True
12  Q4 (highest)        

In [23]:
# Decision (R5): in each GPA group, does failure go down as attendance rises?
results = []
for g, sub in t[t["used"]].groupby("gpa_group", observed=True):
    rates = sub.sort_values("attendance_group")["rate"].tolist()
    if len(rates) < 2:
        continue  # need at least two attendance groups to see a trend
    goes_down = all(later < earlier for earlier, later in zip(rates, rates[1:]))
    results.append((g, len(rates), goes_down))

In [28]:
decision = pd.DataFrame(results, columns=["gpa_group", "attendance_groups_used", "goes_down"])
print("\n" + decision.to_string(index=False))

down = int(decision["goes_down"].sum())
print(f"\nGPA groups used: {len(decision)}; failure goes down with attendance in {down}")
print("Planned rule result:", "MET" if down > len(decision) / 2 else "NOT MET")


   gpa_group  attendance_groups_used  goes_down
 Q1 (lowest)                       4       True
          Q2                       4       True
          Q3                       4      False
Q4 (highest)                       4      False
     Missing                       4      False

GPA groups used: 5; failure goes down with attendance in 2
Planned rule result: NOT MET


**Result of the planned rule:**

**Overall:** among referred students, term 3 failure drops as attendance
rises: 31.7% (0–25%), 24.7%, 16.7%, 11.6% (above 75–100%).

Inside GPA groups, failure goes down at every step in only 2 of 5 groups
(Q1 and Q2), so by our planned rule **H2 is rejected as stated**.

Looking at the table, this is because:
- **Q4:** no student failed in any attendance group, so the rate cannot go down.
- **Q3 and Missing:** very few failures (for example 2 vs. 7, and 3 vs. 3), so
  small random bumps break the "every step" rule.
- **Q1 and Q2,** where failure is common, show a clear drop.

**Result of the planned rule:**

**Overall:** among referred students, term 3 failure drops as attendance
rises: 31.7% (0–25%), 24.7%, 16.7%, 11.6% (above 75–100%).

Inside GPA groups, failure goes down at every step in only 2 of 5 groups
(Q1 and Q2), so by our planned rule **H2 is rejected as stated**.

Looking at the table, this is because:
- **Q4:** no student failed in any attendance group, so the rate cannot go down.
- **Q3 and Missing:** very few failures (for example 2 vs. 7, and 3 vs. 3), so
  small random bumps break the "every step" rule.
- **Q1 and Q2,** where failure is common, show a clear drop.

### Extra check (not the planned rule): lowest vs. highest attendance
The planned rule needs the failure rate to drop at every step. To understand
why it was not met, we also compare only the lowest (0–25%) and highest
(above 75–100%) attendance groups in each GPA group. This check was added
after seeing the result, so it explains the result but does not replace the
planned rule.

In [25]:
# Compare the lowest and highest attendance groups inside each GPA group
low = t[t["attendance_group"] == "0-25%"].set_index("gpa_group")
high = t[t["attendance_group"] == ">75-100%"].set_index("gpa_group")

check = pd.DataFrame({
    "fails_low": low["fails"], "rate_low": low["rate"], "ci_low": low["ci"],
    "fails_high": high["fails"], "rate_high": high["rate"], "ci_high": high["ci"],
})
check["difference"] = check["rate_high"] - check["rate_low"]
print(check.to_string(formatters={"rate_low": "{:.1%}".format,
                                   "rate_high": "{:.1%}".format,
                                   "difference": "{:+.1%}".format}))

              fails_low rate_low          ci_low  fails_high rate_high         ci_high difference
gpa_group                                                                                        
Q1 (lowest)         554    44.3%  41.6% to 47.1%         416     18.7%  17.2% to 20.4%     -25.6%
Q2                   31     8.2%   5.8% to 11.4%          13      1.6%    0.9% to 2.7%      -6.6%
Q3                    2     1.4%    0.4% to 4.8%           0      0.0%    0.0% to 0.8%      -1.4%
Q4 (highest)          0     0.0%    0.0% to 6.4%           0      0.0%    0.0% to 3.0%      +0.0%
Missing              11    19.6%  11.3% to 31.8%           3      2.9%    1.0% to 8.1%     -16.8%


**Result:** In every GPA group where some students failed, the highest
attendance group failed less often than the lowest: Q1 44.3% → 18.7%,
Q2 8.2% → 1.6%, Missing 19.6% → 2.9% (intervals do not overlap), and
Q3 1.4% → 0.0% (too few failures to be sure). Q4 had no failures, so it
cannot show a difference.

**H2 verdict: revised.** The planned rule was not met, but the data supports a
narrower statement: among referred students who were struggling (lower GPA
groups), those who attended more sessions failed less often. Among strong
students, almost no one failed, so attendance could not make a difference we
can measure.

**Caution:** attendance is linked to lower failure, but this does not prove
tutoring caused it. Students who attend more may also be more motivated,
which the data does not measure.

## Step 5: Do bus riders attend fewer tutoring sessions?

H2 showed that referred students who attend more sessions fail less, especially
lower-GPA students. So the next question is what keeps some students from attending.
One possible reason is transportation: students who ride the bus may not be able to
stay after school for tutoring.

This is an extra check, not one of our hypotheses. It describes the data and does not
test a planned rule. We use the same referred students as H2 and compare the attendance
of bus riders and non-riders, overall and inside each prior GPA group.

In [29]:
# Same students as H2: referred before term 3 and had at least one session
bus = h2.copy()
bus["rider"] = bus["bus_rider"].map({1: "Bus rider", 0: "Not a bus rider"})

# Mark students in the lowest attendance group (0-25%)
bus["low_attendance"] = (bus["attendance_group"] == "0-25%").astype(int)

# 1) Overall: bus riders vs. non-riders
overall = bus.groupby("rider").agg(
    n=("student_id", "size"),
    mean_attendance=("attendance_rate_before_t3", "mean"),
    share_low_attendance=("low_attendance", "mean"),
)
print("Overall:")
print(overall.to_string(formatters={"mean_attendance": "{:.1%}".format,
                                    "share_low_attendance": "{:.1%}".format}))

# 2) Inside each prior GPA group, so prior grades don't mix up the comparison
by_gpa = bus.pivot_table(index="gpa_group", columns="bus_rider",
                         values="attendance_rate_before_t3",
                         aggfunc=["size", "mean"], observed=False)
by_gpa.columns = ["n_non_rider", "n_bus_rider", "att_non_rider", "att_bus_rider"]
by_gpa = by_gpa.reindex(GPA_ORDER)
by_gpa["difference"] = by_gpa["att_bus_rider"] - by_gpa["att_non_rider"]

print("\nBy prior GPA group:")
print(by_gpa.to_string(formatters={"att_non_rider": "{:.1%}".format,
                                   "att_bus_rider": "{:.1%}".format,
                                   "difference": "{:+.1%}".format}))

Overall:
                    n mean_attendance share_low_attendance
rider                                                     
Bus rider        8272           53.4%                17.0%
Not a bus rider  5266           62.1%                 9.2%

By prior GPA group:
              n_non_rider  n_bus_rider att_non_rider att_bus_rider difference
gpa_group                                                                    
Q1 (lowest)          3239         5287         61.4%         52.6%      -8.8%
Q2                   1144         1742         62.9%         53.6%      -9.3%
Q3                    559          774         63.7%         58.1%      -5.6%
Q4 (highest)          180          235         63.3%         54.5%      -8.8%
Missing               144          234         62.0%         53.1%      -8.9%


**Result:** Among referred students, bus riders attended fewer tutoring sessions than
non-riders: 53.4% vs. 62.1% on average. They were also almost twice as likely to be in
the lowest attendance group (0-25%): 17.0% vs. 9.2%.

This gap is not explained by prior grades. Inside every prior GPA group, bus riders
attended less, by about 6 to 9 points (for example, Q1: 52.6% vs. 61.4%).

**What this means:** H2 showed that lower-GPA referred students who attend more fail
less. Bus riders attend less, so they may get less benefit from tutoring. This could be
one reason why some referred students still fail.

**Caution:** this is a description, not proof. The data does not say *why* bus riders
attend less. Transportation is our guess. Bus riders may also differ in other ways, for
example family income, which we did not hold fixed here.

## Step 6: Summary of the experiments

| | What we tested | Result | Verdict |
|---|---|---|---|
| **H1** | Inside groups with the same prior GPA and term 1 result, do referred students fail less than non-referred students? | Overall, referred students fail more (19.6% vs. 8.9%). But inside all 7 groups we could use, referred students failed less. | **Supported** |
| **H2** | Among referred students, does failure go down at every step as attendance goes up, inside each prior GPA group? | Failure dropped at every step in only 2 of 5 GPA groups (Q1, Q2). Lowest vs. highest attendance: a clear drop in Q1, Q2 and Missing. Q3 and Q4 had almost no failures. | **Revised** |
| Extra | Do bus riders attend fewer sessions? | Yes, by about 6 to 9 points in every GPA group. | Description only |

**What we learned:**
- The tutoring paradox is a case of **Simpson's paradox**. Students are referred *because*
  they were already struggling, and struggling students fail more. When we compare
  students with the same prior record, referred students do better.
- Attendance matters most for students who were struggling (lower GPA groups).
- Bus riders attend less, so transportation may be a barrier to getting the benefit.

**Limitations:**
- These are comparisons in past data, not an experiment. We cannot prove that tutoring
  *caused* lower failure.
- Things like motivation and parent support are not in the data. They could affect
  both attendance and passing.
- We do not know the reason for each referral, beyond who made it.
- Some groups (Q3, Q4) had very few failures, so their results are less certain.

**Next (models, R7):** we will check whether these findings also hold when predicting.
Does adding prior GPA, referral and attendance help a naive Bayes model predict term 3
failure?